## MlFlow
- So the thing over here is that is nothing

In [1]:
import mlflow
mlflow.set_experiment("Random Forest")

2026/08/04 18:39:43 INFO mlflow.tracking.fluent: Experiment with name 'Random Forest' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/absyd/mystfs/cs/ai/coursingggs_/AI_ENG_FROM_SCRATCH/02-ml-fundamentals/mlruns/1', creation_time=1785847183593, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785847183593, lifecycle_stage='active', name='Random Forest', tags={}, trace_location=None, workspace='default'>

In [2]:
# Everything inside block belongs to one run.
with mlflow.start_run():
    print("Training...")

Training...


In [3]:
import mlflow

mlflow.set_experiment("Titanic")

with mlflow.start_run():

    print("training...")

2026/08/04 18:41:45 INFO mlflow.tracking.fluent: Experiment with name 'Titanic' does not exist. Creating a new experiment.


training...


In [7]:
import os
import mlflow
import mlflow.sklearn

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# # Optional: force local tracking directory
# mlflow.set_tracking_uri("file:./mlruns")

# Create/select experiment
mlflow.set_experiment("Iris Classification")

# Dataset
X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Pipeline
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42
    ))
])

# Start tracking
with mlflow.start_run() as run:

    # Parameters
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("random_state", 42)

    # Train
    pipe.fit(X_train, y_train)

    # Evaluate
    accuracy = pipe.score(X_test, y_test)

    # Metric
    mlflow.log_metric("accuracy", accuracy)

    # Save model
    mlflow.sklearn.log_model(pipe, "model")

    print("Run ID:", run.info.run_id)
    print("Accuracy:", accuracy)

# print("Tracking URI:", mlflow.get_tracking_uri())
print("Current directory:", os.getcwd())

MlflowException: The filesystem tracking backend (e.g., './mlruns') is in maintenance mode and will not receive further updates. Please migrate to a database backend (e.g., 'sqlite:///mlflow.db') to access the latest MLflow features. The `mlflow migrate-filestore` tool migrates your existing data losslessly. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance. If the filesystem backend is required for your workflow, set `MLFLOW_ALLOW_FILE_STORE=true` to opt out of this exception.

In [5]:
import mlflow
import os

print(mlflow.__version__)
print(mlflow.get_tracking_uri())
print(os.getcwd())


3.15.1
file:./mlruns
/Users/absyd/mystfs/cs/ai/coursingggs_/AI_ENG_FROM_SCRATCH/02-ml-fundamentals


In [11]:
import os

# Must come BEFORE importing mlflow
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:///mlflow.db"

import mlflow
import mlflow.sklearn

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Tracking backend
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# Experiment
mlflow.set_experiment("Iris Classification")

# Dataset
X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

# Pipeline
pipe = Pipeline([
    ("scaler", StandardScaler()),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=100,
            max_depth=5,
            random_state=42,
        ),
    ),
])

# Start run
with mlflow.start_run() as run:

    # Parameters
    mlflow.log_params({
        "n_estimators": 100,
        "max_depth": 5,
        "random_state": 42,
        "test_size": 0.2,
    })

    # Train
    pipe.fit(X_train, y_train)

    # Metric
    accuracy = pipe.score(X_test, y_test)

    mlflow.log_metric("accuracy", accuracy)

    # Tags
    mlflow.set_tags({
        "author": "Abu Sayed",
        "dataset": "Iris",
        "framework": "Scikit-learn",
    })

    # Save model
    mlflow.sklearn.log_model(
        sk_model=pipe,
        name="random_forest_model",
    )

    print("=" * 50)
    print("Run ID      :", run.info.run_id)
    print("Experiment  :", mlflow.get_experiment(run.info.experiment_id).name)
    print("Accuracy    :", accuracy)
    print("=" * 50)

print("\nTracking URI :", mlflow.get_tracking_uri())
print("Working Dir  :", os.getcwd())
print("Database     : mlflow.db")
print("Artifacts    : ./mlartifacts")

2026/08/04 18:58:16 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Run ID      : f8062cbbb938431da89c3c98bcc637c9
Experiment  : Iris Classification
Accuracy    : 1.0

Tracking URI : sqlite:///mlflow.db
Working Dir  : /Users/absyd/mystfs/cs/ai/coursingggs_/AI_ENG_FROM_SCRATCH/02-ml-fundamentals
Database     : mlflow.db
Artifacts    : ./mlartifacts


In [12]:
!mlflow ui --backend-store-uri sqlite:///mlflow.db

Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
2026/08/04 18:58:31 INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
2026/08/04 18:58:31 INFO:     Started parent process [72611]
/Users/absyd/mystfs/cs/ai/coursingggs_/AI_ENG_FROM_SCRATCH/.venv/lib/python3.12/site-packages/mlflow/server/fastapi_app.py:19: StarletteDeprecationWarning: starlette.middleware.wsgi is deprecated and will be removed in a future release. Please refer to https://github.com/abersheeran/a2wsgi as a replacement.
  from starlette.middleware.wsgi import WSGIResponder, build_environ
/Users/absyd/mystfs/cs/ai/coursingggs_/AI_ENG_FROM_SCRATCH/.venv/lib/python3.12/site-packages/mlflow/server/fastapi_app.py:19: StarletteDeprecationWarning: starlette.middleware.wsgi is deprecated and will be removed 

In [10]:
print(mlflow.get_tracking_uri())

sqlite:///mlflow.db


In [13]:
import mlflow

print(mlflow.get_tracking_uri())

sqlite:///mlflow.db


In [14]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")

experiments = mlflow.search_experiments()

print(experiments)

runs = mlflow.search_runs()

print(runs)

[<Experiment: artifact_location='/Users/absyd/mystfs/cs/ai/coursingggs_/AI_ENG_FROM_SCRATCH/02-ml-fundamentals/mlruns/3', creation_time=1785847881061, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1785847881061, lifecycle_stage='active', name='Iris Classification', tags={}, trace_location=None, workspace='default'>, <Experiment: artifact_location='/Users/absyd/mystfs/cs/ai/coursingggs_/AI_ENG_FROM_SCRATCH/02-ml-fundamentals/mlruns/2', creation_time=1785847305624, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1785847305624, lifecycle_stage='active', name='Titanic', tags={}, trace_location=None, workspace='default'>, <Experiment: artifact_location='/Users/absyd/mystfs/cs/ai/coursingggs_/AI_ENG_FROM_SCRATCH/02-ml-fundamentals/mlruns/1', creation_time=1785847183593, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785847183593, lifecycle_stage='active', name='Random Forest', tags={}, trace_location=N

In [15]:
from pathlib import Path
import mlflow

print("URI:", mlflow.get_tracking_uri())
print("DB exists:", Path("mlflow.db").exists())

print(mlflow.search_experiments())
print(mlflow.search_runs())

URI: sqlite:///mlflow.db
DB exists: True
[<Experiment: artifact_location='/Users/absyd/mystfs/cs/ai/coursingggs_/AI_ENG_FROM_SCRATCH/02-ml-fundamentals/mlruns/3', creation_time=1785847881061, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1785847881061, lifecycle_stage='active', name='Iris Classification', tags={}, trace_location=None, workspace='default'>, <Experiment: artifact_location='/Users/absyd/mystfs/cs/ai/coursingggs_/AI_ENG_FROM_SCRATCH/02-ml-fundamentals/mlruns/2', creation_time=1785847305624, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1785847305624, lifecycle_stage='active', name='Titanic', tags={}, trace_location=None, workspace='default'>, <Experiment: artifact_location='/Users/absyd/mystfs/cs/ai/coursingggs_/AI_ENG_FROM_SCRATCH/02-ml-fundamentals/mlruns/1', creation_time=1785847183593, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785847183593, lifecycle_stage='active', name='